In [ ]:
!nvidia-smi



Sun Jun  7 08:17:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter


Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp6_h8c7p0".


In [ ]:
!pip install pycuda


In [ ]:
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# 1. Define the CUDA C++ kernel as a Python string
cuda_code = """
__global__ void vectorAdd(const float *A, const float *B, float *C, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}
"""

# 2. Compile the CUDA kernel code
mod = SourceModule(cuda_code)
vectorAdd = mod.get_function("vectorAdd")

# 3. Create host data (NumPy arrays)
# Note: Modern GPUs require float32 for standard single-precision operations
N = 1000
h_A = np.ones(N, dtype=np.float32)      # Array of 1.0s
h_B = np.ones(N, dtype=np.float32) * 2  # Array of 2.0s
h_C = np.zeros(N, dtype=np.float32)     # Array to hold the result

# 4. Allocate memory on the GPU (Device)
d_A = cuda.mem_alloc(h_A.nbytes)
d_B = cuda.mem_alloc(h_B.nbytes)
d_C = cuda.mem_alloc(h_C.nbytes)

# 5. Copy data from Host (CPU) to Device (GPU)
cuda.memcpy_htod(d_A, h_A)
cuda.memcpy_htod(d_B, h_B)

# 6. Define block and grid dimensions
threads_per_block = 256
blocks_per_grid = int(np.ceil(N / threads_per_block))

# 7. Launch the kernel
# We pass the GPU pointers, the size N (as an int32), block size, and grid size
vectorAdd(
    d_A, d_B, d_C, np.int32(N),
    block=(threads_per_block, 1, 1),
    grid=(blocks_per_grid, 1)
)

# 8. Copy the result back from Device (GPU) to Host (CPU)
cuda.memcpy_dtoh(h_C, d_C)

# 9. Verify the results
if np.allclose(h_C, 3.0):
    print(f"Success! All {N} vector elements were added correctly via PyCUDA.")
else:
    print("Error: Result mismatch.")

Success! All 1000 vector elements were added correctly via PyCUDA.


In [ ]:
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import os

# =====================================================================
# 1. GENERATE A MOCK FILE (For demonstration purposes)
# =====================================================================
filename = "sensor_data.txt"
N = 1000000  # 1 Million sensor data points

print("Generating 1 million data points into a file...")
# Create random data between 10.0 and 80.0. Inject a few anomalies above 95.0
np.random.seed(42)
mock_data = np.random.uniform(10.0, 80.0, N).astype(np.float32)
mock_data[1500] = 98.4    # Anomaly 1
mock_data[500000] = 102.1  # Anomaly 2
mock_data[999999] = 96.7  # Anomaly 3

# Save to a plain text file (one float per line)
np.savetxt(filename, mock_data, fmt='%f')
print(f"File '{filename}' created successfully.\n")


# =====================================================================
# 2. READ THE FILE INTO PYTHON
# =====================================================================
print("Python: Reading file into memory...")
# Read the file back into a NumPy float32 array
data_from_file = np.loadtxt(filename, dtype=np.float32)


# =====================================================================
# 3. DEFINE AND COMPILE THE CUDA KERNEL
# =====================================================================
# This kernel checks if a reading is above a threshold.
# If it is, it writes a '1' (True) into the output array; otherwise '0'.
cuda_code = """
__global__ void analyzeData(const float *data, int *anomalies, float threshold, int N) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    if (idx < N) {
        if (data[idx] > threshold) {
            anomalies[idx] = 1; // Mark as anomaly
        } else {
            anomalies[idx] = 0; // Normal
        }
    }
}
"""
mod = SourceModule(cuda_code)
analyze_kernel = mod.get_function("analyzeData")


# =====================================================================
# 4. MEMORY MANAGEMENT & GPU EXECUTION
# =====================================================================
# Create a matching output array for the GPU to write its results into
output_results = np.zeros(N, dtype=np.int32)

print("GPU: Allocating VRAM and transferring data...")
# Allocate memory on the GPU
d_data = cuda.mem_alloc(data_from_file.nbytes)
d_output = cuda.mem_alloc(output_results.nbytes)

# Copy the file data from CPU RAM to GPU VRAM
cuda.memcpy_htod(d_data, data_from_file)

# Configure execution grid (Blocks of 256 threads)
threads_per_block = 256
blocks_per_grid = int(np.ceil(N / threads_per_block))

print("GPU: Analyzing data in parallel...")
# Run the kernel (Threshold set to 95.0)
threshold_val = np.float32(95.0)
analyze_kernel(
    d_data, d_output, threshold_val, np.int32(N),
    block=(threads_per_block, 1, 1),
    grid=(blocks_per_grid, 1)
)

# Copy the classification results back to CPU memory
cuda.memcpy_dtoh(output_results, d_output)


# =====================================================================
# 5. POST-PROCESSING THE RESULTS IN PYTHON
# =====================================================================
# Use NumPy to instantly find the indices where the GPU flagged a '1'
anomaly_indices = np.where(output_results == 1)[0]

print("\n================ ANALYSIS RESULTS ================")
print(f"Total data points analyzed: {N}")
print(f"Anomalies detected: {len(anomaly_indices)}")

for idx in anomaly_indices:
    print(f"-> Anomaly found at line {idx + 1}: Value = {data_from_file[idx]}")

Generating 1 million data points into a file...
File 'sensor_data.txt' created successfully.

Python: Reading file into memory...
GPU: Allocating VRAM and transferring data...
GPU: Analyzing data in parallel...

================ ANALYSIS RESULTS ================
Total data points analyzed: 1000000
Anomalies detected: 3
-> Anomaly found at line 1501: Value = 98.4000015258789
-> Anomaly found at line 500001: Value = 102.0999984741211
-> Anomaly found at line 1000000: Value = 96.69999694824219


In [ ]:
import pandas as pd
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# =====================================================================
# 0. GENERATE A MOCK FILE (For demonstration purposes)
# =====================================================================
print("Generating mock EV.txt file...")
num_stations = 100
mock_df = pd.DataFrame({
    "Station ID": [f"EV{i:03d}" for i in range(num_stations)],
    "Address": [f"Address {i}" for i in range(num_stations)],
    "Charger Type": np.random.choice(["Level 2", "DC Fast"], num_stations),
    "Cost (USD/kWh)": np.random.uniform(0.20, 0.50, num_stations).round(2),
    "Usage Stats (avg users/day)": np.random.randint(5, 100, num_stations),
    "Charging Capacity (kW)": np.random.uniform(7, 150, num_stations).round(1)
})
mock_df.to_csv("EV.txt", index=False)
print("Mock EV.txt created successfully.\n")

# =====================================================================
# 1. PARSE & SANITIZE THE INPUT FILE (CPU BOUND)
# =====================================================================
print("Step 1: Python reading and clean-parsing EV.txt...")

# Using pandas to cleanly extract numeric data out of the complex strings
df = pd.read_csv("EV.txt", skipinitialspace=True)

# We want to extract 3 core columns to calculate dynamic station metrics
# Columns: Cost (USD/kWh), Usage Stats (avg users/day), Charging Capacity (kW)
cost_data     = df["Cost (USD/kWh)"].values.astype(np.float32)
usage_data    = df["Usage Stats (avg users/day)"].values.astype(np.float32)
c_capacity_data = df["Charging Capacity (kW)"].values.astype(np.float32)

# Get the total number of records (N)
N = len(df)
print(f"-> Successfully extracted numerical fields for {N} EV stations.\n")


# =====================================================================
# 2. WRITE AND COMPILE THE CUDA KERNEL (GPU JIT)
# =====================================================================
# This custom C++ parallel kernel multiplies the 3 metrics together
# for every station simultaneously to calculate dynamic daily revenue metrics.
cuda_code = """
__global__ void analyzeEVRevenue(const float *cost, const float *usage, const float *capacity, float *revenue, int N) {
    // Calculate the global hardware Thread Index
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    // Boundary check to ensure we don't access beyond the array limit
    if (idx < N) {
        // Daily Station Revenue = Cost/kWh * Avg Daily Users * Charging Capacity (kW pool)
        revenue[idx] = cost[idx] * usage[idx] * capacity[idx];
    }
}
"""
print("Step 2: Compiling custom C++ CUDA kernel via nvcc...")
mod = SourceModule(cuda_code)
analyze_kernel = mod.get_function("analyzeEVRevenue")


# =====================================================================
# 3. GPU VRAM ALLOCATION & DATA SHIFTING
# =====================================================================
# Creating an empty array in system RAM to host the final results
revenue_results = np.zeros(N, dtype=np.float32)

print("Step 3: Allocating separate VRAM spaces and shifting data (Host -> Device)...")
# Allocate raw bytes inside the GPU
d_cost     = cuda.mem_alloc(cost_data.nbytes)
d_usage    = cuda.mem_alloc(usage_data.nbytes)
d_capacity = cuda.mem_alloc(c_capacity_data.nbytes)
d_revenue  = cuda.mem_alloc(revenue_results.nbytes)

# Physcially transfer the raw parsed file arrays into the GPU architecture
cuda.memcpy_htod(d_cost, cost_data)
cuda.memcpy_htod(d_usage, usage_data)
cuda.memcpy_htod(d_capacity, c_capacity_data)


# =====================================================================
# 4. EXECUTE CUDA PARALLEL GRID
# =====================================================================
# Group threads into typical blocks of 256
threads_per_block = 256
blocks_per_grid = int(np.ceil(N / threads_per_block))

print(f"Step 4: Spawning {blocks_per_grid} blocks of {threads_per_block} threads on GPU...")
# Run the kernel across thousands of micro-cores simultaneously
analyze_kernel(
    d_cost, d_usage, d_capacity, d_revenue, np.int32(N),
    block=(threads_per_block, 1, 1),
    grid=(blocks_per_grid, 1)
)

# Fetch the processed output back out of the GPU (Device -> Host)
cuda.memcpy_dtoh(revenue_results, d_revenue)
print("-> GPU computing complete! Shifted results back to Python environment.\n")


# =====================================================================
# 5. MERGE RESULTS & LOG HIGH LEVEL INSIGHTS
# =====================================================================
# Inject our GPU results directly back into our structured dataframe
df["Projected_Daily_Revenue_USD"] = revenue_results

print("================== EV ACCELERATED ANALYSIS REPORT ==================")
print(f"Processed Stations Total: {N}")
print(f"Average Projected Revenue/Station/Day: ${df['Projected_Daily_Revenue_USD'].mean():.2f}")
print(f"Highest Grossing Single Station:       ${df['Projected_Daily_Revenue_USD'].max():.2f}")

print("\nTop 5 Most Economically Profitable Stations Found:")
top_5 = df.sort_values(by="Projected_Daily_Revenue_USD", ascending=False).head(5)
for index, row in top_5.iterrows():
    print(f"  -> Station {row['Station ID']} ({row['Charger Type']}) located at {row['Address']}: "
          f"Generated ${row['Projected_Daily_Revenue_USD']:.2f}/day")

Generating mock EV.txt file...
Mock EV.txt created successfully.

Step 1: Python reading and clean-parsing EV.txt...
-> Successfully extracted numerical fields for 100 EV stations.

Step 2: Compiling custom C++ CUDA kernel via nvcc...
Step 3: Allocating separate VRAM spaces and shifting data (Host -> Device)...
Step 4: Spawning 1 blocks of 256 threads on GPU...
-> GPU computing complete! Shifted results back to Python environment.

================== EV ACCELERATED ANALYSIS REPORT ==================
Processed Stations Total: 100
Average Projected Revenue/Station/Day: $1612.78
Highest Grossing Single Station:       $5607.15

Top 5 Most Economically Profitable Stations Found:
  -> Station EV027 (DC Fast) located at Address 27: Generated $5607.15/day
  -> Station EV063 (Level 2) located at Address 63: Generated $5143.74/day
  -> Station EV013 (DC Fast) located at Address 13: Generated $4866.05/day
  -> Station EV096 (Level 2) located at Address 96: Generated $4454.37/day
  -> Station EV03